# glcuda Wave 102 — N16 register-prefetch production Q8 gate

Hard-selected Tesla T4, byte-locked Q8 model, two direct gates, and ten position-balanced production pairs.


In [ ]:
import base64
import gzip
import hashlib
import json
import math
import os
from pathlib import Path
import re
import shutil
import subprocess
import traceback
import urllib.request
import zipfile

BUILD = "wave102-n16-prefetch-v2"
REPO_URL = "https://github.com/gwenland-org/gwenland-ai.git"
BASE_REV = "bd5c956bafb3bb6738c3f1de348a4ebb55d9c29f"
SOURCE_REV = "ec4a6332558070cb6b3c6371d3bdacb8f18b6d78"
PATCH_SHA256 = "272ae9add1a8b1b9f56ff4bee97d95aad2fdb461c29f2224388257896cae0279"
PATCH_GZIP_B64 = """..."""
ROOT = Path("/kaggle/working/wave102")
TREE = ROOT / "repo"
RESULTS = ROOT / "results"
TARGET = ROOT / "target"
FINAL_ZIP = Path("/kaggle/working/glcuda-t4-wave102-n16-prefetch-production-results.zip")

if ROOT.exists():
    shutil.rmtree(ROOT)
RESULTS.mkdir(parents=True)


def run(cmd, *, cwd=None, env=None, timeout=7200, check=True):
    merged = os.environ.copy()
    if env:
        merged.update({k: str(v) for k, v in env.items()})
    p = subprocess.run(
        [str(x) for x in cmd], cwd=cwd, env=merged, text=True,
        capture_output=True, timeout=timeout,
    )
    print("$", " ".join(str(x) for x in cmd), flush=True)
    if p.stdout:
        print(p.stdout[-12000:], flush=True)
    if p.stderr:
        print(p.stderr[-12000:], flush=True)
    if check and p.returncode:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p


def save(name, p):
    (RESULTS / name).write_text(
        f"RETURN_CODE {p.returncode}\n\nSTDOUT\n{p.stdout}\n\nSTDERR\n{p.stderr}",
        encoding="utf-8",
    )


def archive():
    if FINAL_ZIP.exists():
        FINAL_ZIP.unlink()
    with zipfile.ZipFile(FINAL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
        for path in sorted(RESULTS.rglob("*")):
            if path.is_file():
                z.write(path, path.relative_to(RESULTS))
    digest = hashlib.sha256(FINAL_ZIP.read_bytes()).hexdigest()
    print("ARCHIVE", FINAL_ZIP, digest, flush=True)
    return digest


def fail(phase):
    (RESULTS / "FAILED.json").write_text(
        json.dumps({"phase": phase, "traceback": traceback.format_exc()}, indent=2),
        encoding="utf-8",
    )
    archive()
    raise RuntimeError(f"Wave 102 failed in {phase}")


def resource(log, entry):
    marker = re.search(
        r"Compiling entry function ['\"]" + re.escape(entry) +
        r"['\"].*?(?=Compiling entry function|\Z)", log, re.S,
    )
    segment = marker.group(0) if marker else ""
    number = lambda pattern: [int(x) for x in re.findall(pattern, segment)]
    regs = number(r"Used (\d+) registers")
    smem = number(r"(\d+) bytes smem")
    return {
        "entry": entry,
        "found": bool(marker),
        "registers": regs[0] if regs else None,
        "static_shared_bytes": smem[0] if smem else 0,
        "stack_frame_bytes": max(number(r"(\d+) bytes stack frame"), default=0),
        "spill_store_bytes": max(number(r"(\d+) bytes spill stores"), default=0),
        "spill_load_bytes": max(number(r"(\d+) bytes spill loads"), default=0),
    }


phase = "bootstrap"
try:
    patch = gzip.decompress(base64.b64decode(PATCH_GZIP_B64))
    got = hashlib.sha256(patch).hexdigest()
    if got != PATCH_SHA256:
        raise RuntimeError(f"patch hash mismatch: {got} != {PATCH_SHA256}")
    patch_path = RESULTS / "wave94.patch"
    patch_path.write_bytes(patch)
    (RESULTS / "source.json").write_text(json.dumps({
        "build": BUILD, "base_rev": BASE_REV, "source_rev": SOURCE_REV,
        "patch_sha256": PATCH_SHA256, "patch_bytes": len(patch),
    }, indent=2), encoding="utf-8")

    gpu = run([
        "nvidia-smi", "--query-gpu=index,name,compute_cap,memory.total,driver_version",
        "--format=csv,noheader,nounits",
    ], timeout=60)
    save("nvidia-smi.log", gpu)
    fields = [x.strip() for x in gpu.stdout.splitlines()[0].split(",")]
    if len(fields) < 5 or fields[1] != "Tesla T4" or fields[2] != "7.5":
        raise RuntimeError(f"Wave 102 requires Tesla T4 sm_75, got {fields}")

    phase = "reconstruct"
    clone = run(["git", "clone", "--filter=blob:none", REPO_URL, TREE], timeout=1800)
    save("git-clone.log", clone)
    checkout = run(["git", "checkout", "--detach", BASE_REV], cwd=TREE, timeout=600)
    save("git-checkout.log", checkout)
    applied = run(["git", "apply", "--whitespace=error", patch_path], cwd=TREE)
    save("git-apply.log", applied)
    diff_check = run(["git", "diff", "--check"], cwd=TREE)
    save("git-diff-check.log", diff_check)

    phase = "ptxas-resource"
    ptxas = shutil.which("ptxas") or "/usr/local/cuda/bin/ptxas"
    if not Path(ptxas).is_file():
        raise RuntimeError(f"ptxas unavailable: {ptxas}")
    retained_ptx = TREE / "glcuda/src/kernels/glcuda_sm75.ptx"
    candidate_ptx = TREE / "glcuda/src/kernels/glcuda_sm75_wave88.ptx"
    p_retained = run([ptxas, "-v", "-arch=sm_75", retained_ptx,
                      "-o", ROOT / "retained.cubin"], cwd=TREE, timeout=1800)
    save("ptxas-retained.log", p_retained)
    p_candidate = run([ptxas, "-v", "-arch=sm_75", candidate_ptx,
                       "-o", ROOT / "candidate.cubin"], cwd=TREE, timeout=1800)
    save("ptxas-candidate.log", p_candidate)
    retained = resource(p_retained.stdout + "\n" + p_retained.stderr,
                        "gl_gemm_mma_q8_bstage_n16")
    candidate = resource(p_candidate.stdout + "\n" + p_candidate.stderr,
                         "gl_gemm_mma_q8_bstage_n16_prefetch")
    resources = {"retained": retained, "candidate": candidate}
    (RESULTS / "resources.json").write_text(
        json.dumps(resources, indent=2), encoding="utf-8"
    )
    if not retained["found"] or not candidate["found"]:
        raise RuntimeError(f"resource entry missing: {resources}")
    if retained["registers"] > 72 or candidate["registers"] > 80:
        raise RuntimeError(f"register gate failed: {resources}")
    if retained["static_shared_bytes"] != 9728 or candidate["static_shared_bytes"] != 9728:
        raise RuntimeError(f"shared-memory gate failed: {resources}")
    for row in resources.values():
        if row["stack_frame_bytes"] or row["spill_store_bytes"] or row["spill_load_bytes"]:
            raise RuntimeError(f"stack/spill gate failed: {resources}")

    phase = "build-test"
    cargo_candidates = [
        shutil.which("cargo"),
        Path.home() / ".cargo/bin/cargo",
        "/usr/local/cargo/bin/cargo",
        "/opt/rust/bin/cargo",
        "/opt/conda/bin/cargo",
        "/usr/local/bin/cargo",
        "/usr/bin/cargo",
    ]
    cargo = next(
        (str(path) for path in cargo_candidates if path and Path(path).is_file()),
        None,
    )
    cargo_env = {}
    bootstrapped = False
    if cargo is None:
        bootstrapped = True
        rustup_url = "https://sh.rustup.rs"
        rustup_script = ROOT / "rustup-init.sh"
        with urllib.request.urlopen(rustup_url, timeout=120) as response:
            rustup_script.write_bytes(response.read())
        cargo_home = ROOT / "cargo-home"
        rustup_home = ROOT / "rustup-home"
        cargo_env = {
            "CARGO_HOME": cargo_home,
            "RUSTUP_HOME": rustup_home,
        }
        install = run(
            ["bash", rustup_script, "-y", "--profile", "minimal",
             "--default-toolchain", "stable", "--no-modify-path"],
            env=cargo_env, timeout=1800,
        )
        save("rustup-install.log", install)
        cargo = str(cargo_home / "bin/cargo")
    if not Path(cargo).is_file():
        raise RuntimeError(f"cargo unavailable after discovery/bootstrap: {cargo}")
    (RESULTS / "cargo-discovery.json").write_text(json.dumps({
        "selected": cargo,
        "bootstrapped": bootstrapped,
        "candidates": [str(path) for path in cargo_candidates if path],
    }, indent=2), encoding="utf-8")
    cargo_version = run([cargo, "--version"], env=cargo_env, timeout=60)
    save("cargo-version.log", cargo_version)
    common = {
        **cargo_env,
        "CARGO_TARGET_DIR": TARGET,
        "CUDA_VISIBLE_DEVICES": "0",
    }
    tests = run([cargo, "test", "-p", "glcuda", "--lib", "--locked"],
                cwd=TREE, env=common)
    save("cargo-lib-tests.log", tests)
    summary = next((x for x in (tests.stdout + tests.stderr).splitlines()
                    if x.startswith("test result:")), "")
    if "65 passed" not in summary or "0 failed" not in summary:
        raise RuntimeError(f"unexpected host test summary: {summary}")
    build = run([cargo, "build", "--release", "-p", "glcuda", "--example",
                 "wave93_n16_prefetch", "--locked"], cwd=TREE, env=common)
    save("cargo-build.log", build)

    phase = "direct-run-1"
    env = {
        **common,
        "GLCUDA_GRID2D": "1",
        "GLCUDA_NTILE128": "1",
        "GLCUDA_BSTAGE": "1",
        "GLCUDA_GEMM_N16": "1",
        "GLCUDA_GEMM_N16_PREFETCH": "1",
    }
    exe = TARGET / "release/examples/wave93_n16_prefetch"
    records = []
    for index in (1, 2):
        phase = f"direct-run-{index}"
        measured = run([exe], cwd=TREE, env=env, check=False)
        save(f"direct-run-{index}.log", measured)
        direct_line = next((x for x in measured.stdout.splitlines()
                            if x.startswith("[wave93-direct] ")), "")
        resource_line = next((x for x in measured.stdout.splitlines()
                              if x.startswith("[wave93-resource] ")), "")
        direct = json.loads(direct_line.split("] ", 1)[1]) if direct_line else {}
        driver = json.loads(resource_line.split("] ", 1)[1]) if resource_line else {}
        records.append({"run": index, "direct": direct, "driver": driver,
                        "returncode": measured.returncode})
        if measured.returncode or direct.get("pass") is not True or direct.get("bit_exact") is not True:
            raise RuntimeError(f"direct run {index} gate failed: {records[-1]}")
        if driver.get("retained_active_blocks_per_sm", 0) < 3 or driver.get("candidate_active_blocks_per_sm", 0) < 3:
            raise RuntimeError(f"driver occupancy gate failed: {records[-1]}")
    (RESULTS / "direct-results.json").write_text(
        json.dumps(records, indent=2), encoding="utf-8"
    )
    verdict = {
        "pass": True,
        "runs": len(records),
        "minimum_speedup": min(x["direct"]["speedup"] for x in records),
        "all_bit_exact": all(x["direct"]["bit_exact"] for x in records),
        "resources": resources,
    }
    (RESULTS / "verdict.json").write_text(
        json.dumps(verdict, indent=2), encoding="utf-8"
    )
    print("WAVE102_VERDICT", json.dumps(verdict), flush=True)
    archive()
except Exception:
    fail(phase)


In [ ]:

# Wave 101 direct gate leaves these authoritative values in scope.
gpu_line = gpu.stdout.splitlines()[0]
direct_result = verdict

import math
import statistics
import time

HF_REPO = "Qwen/Qwen2.5-0.5B-Instruct-GGUF"
HF_REVISION = "9217f5db79a29953eb74d5343926648285ec7e67"
HF_FILENAME = "qwen2.5-0.5b-instruct-q8_0.gguf"
HF_EXPECTED_BYTES = 675710816
HF_EXPECTED_SHA256 = "ca59ca7f13d0e15a8cfa77bd17e65d24f6844b554a7b6c12e07a5f89ff76844e"
PRODUCTION_REPEATS = 10
COLD_ITERS = 1
WARMUP_ITERS = 5
MEASURE_ITERS = 10
FIXED_PROMPT = (
    "Measure this deterministic systems prompt carefully. Explain how token-parallel "
    "integer matrix multiplication uses shared memory, Tensor Cores, and fixed launch geometry. "
) * 8


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def fetch_pinned_model():
    model = ROOT / HF_FILENAME
    part = ROOT / f"{HF_FILENAME}.part"
    if (
        model.is_file()
        and model.stat().st_size == HF_EXPECTED_BYTES
        and sha256_file(model) == HF_EXPECTED_SHA256
    ):
        return model
    if model.exists():
        model.unlink()
    if part.exists() and part.stat().st_size > HF_EXPECTED_BYTES:
        part.unlink()
    url = f"https://huggingface.co/{HF_REPO}/resolve/{HF_REVISION}/{HF_FILENAME}?download=true"
    for attempt in range(1, 6):
        start = part.stat().st_size if part.exists() else 0
        headers = {"User-Agent": "GwenLand-glcuda-Wave102/1.0", "Accept-Encoding": "identity"}
        if start:
            headers["Range"] = f"bytes={start}-"
        try:
            response = urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=120)
            status = getattr(response, "status", response.getcode())
            if start and status != 206:
                response.close()
                part.unlink(missing_ok=True)
                start = 0
                response = urllib.request.urlopen(
                    urllib.request.Request(
                        url,
                        headers={
                            "User-Agent": "GwenLand-glcuda-Wave102/1.0",
                            "Accept-Encoding": "identity",
                        },
                    ),
                    timeout=120,
                )
                status = getattr(response, "status", response.getcode())
            if status not in (200, 206):
                raise RuntimeError(f"HTTP {status}")
            mode = "ab" if start and status == 206 else "wb"
            downloaded = start
            last_print = time.monotonic()
            with response, part.open(mode) as output:
                while True:
                    chunk = response.read(8 << 20)
                    if not chunk:
                        break
                    output.write(chunk)
                    downloaded += len(chunk)
                    if time.monotonic() - last_print >= 20:
                        print(
                            f"model fetch {downloaded / (1 << 20):.1f}/"
                            f"{HF_EXPECTED_BYTES / (1 << 20):.1f} MiB",
                            flush=True,
                        )
                        last_print = time.monotonic()
            if part.stat().st_size != HF_EXPECTED_BYTES:
                raise RuntimeError(f"truncated model: {part.stat().st_size}/{HF_EXPECTED_BYTES}")
            actual = sha256_file(part)
            if actual != HF_EXPECTED_SHA256:
                part.unlink(missing_ok=True)
                raise RuntimeError(f"model SHA mismatch: {actual}")
            part.replace(model)
            return model
        except Exception as exc:
            print(f"fetch attempt {attempt}/5 failed: {exc}", flush=True)
            if attempt == 5:
                raise
            time.sleep(min(30, 2 ** attempt))


def percentile(values, quantile):
    values = sorted(values)
    index = (len(values) - 1) * quantile
    lo, hi = math.floor(index), math.ceil(index)
    return values[lo] if lo == hi else values[lo] * (hi - index) + values[hi] * (index - lo)


def json_lines(haystack, prefix):
    return [
        json.loads(item)
        for item in re.findall(re.escape(prefix) + r"\s*(\{[^\n]+\})", haystack)
    ]


def last_json_line(haystack, prefix):
    matches = json_lines(haystack, prefix)
    if not matches:
        raise RuntimeError(f"dispatch line missing: {prefix}")
    return matches[-1]


def check_dispatch(arm, haystack):
    contract = last_json_line(haystack, "[glcuda-contract]")
    candidate = arm == "candidate_n16_prefetch"
    required = {
        "exact_fusion": True,
        "grid2d": True,
        "ntile128": True,
        "bstage": True,
        "gemm_n16": True,
        "gemm_n32": False,
        "attn_mma4": True,
        "attn_mma4_regq": True,
        "gemm_n16_prefetch": candidate,
        "attn_mma4_av": True,
    }
    bad = {
        key: (contract.get(key), expected)
        for key, expected in required.items()
        if contract.get(key) is not expected
    }
    if bad:
        raise RuntimeError(f"{arm} contract mismatch: {bad}; full={contract}")
    attention = last_json_line(haystack, "[glcuda-attn]")
    expected_attention = "mma4-regq-avmma"
    if attention.get("path") != expected_attention or attention.get("ntok") != 244:
        raise RuntimeError(f"{arm} attention dispatch drift: {attention}")
    gemm = json_lines(haystack, "[glcuda-gemm]")
    paths = {row.get("path") for row in gemm}
    expected_paths = {
        "bstage-n16-m32",
        "bstage-n16-prefetch" if candidate else "bstage-n16",
    }
    if paths != expected_paths or any(
        row.get("ntok") != 244 for row in gemm
    ):
        raise RuntimeError(f"{arm} GEMM dispatch drift: {gemm}")
    return {"contract": contract, "attention": attention, "gemm": gemm}


def timing_rows(rows, expected, label):
    if len(rows) != expected:
        raise RuntimeError(f"{label} count {len(rows)} != {expected}")
    counts = [int(row.get("prompt_tokens", 0)) for row in rows]
    prefill = [float(row.get("prefill_ms", 0)) for row in rows]
    decode = [float(row.get("decode_ms", 0)) for row in rows]
    if len(set(counts)) != 1 or counts[0] != 244:
        raise RuntimeError(f"{label} prompt-token drift: {counts}")
    if not all(math.isfinite(value) and value > 0 for value in prefill + decode):
        raise RuntimeError(f"{label} malformed timings: {rows}")
    return counts[0], prefill, decode


def session_stats(path):
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    engine = data.get("engine") or {}
    workload = data.get("workload") or {}
    if engine.get("name") != "glcuda" or engine.get("backend") != "cuda" or not engine.get("available"):
        raise RuntimeError(f"wrong engine contract: {engine}")
    expected = {
        "engine": "glcuda",
        "kind": "prefill",
        "prompt": FIXED_PROMPT,
        "seed": 42,
        "temperature": 0.0,
        "max_new_tokens": 1,
        "cold_iters": COLD_ITERS,
        "warmup_iters": WARMUP_ITERS,
        "measure_iters": MEASURE_ITERS,
        "verify_against": "glproc",
    }
    for key, value in expected.items():
        if workload.get(key) != value:
            raise RuntimeError(f"workload {key} mismatch: {workload.get(key)!r} != {value!r}")
    validation = data.get("validation") or {}
    parity = [
        finding for finding in validation.get("findings", [])
        if finding.get("check") == "parity"
    ]
    match = re.search(
        r"(\d+)/(\d+) tokens match oracle",
        parity[-1].get("message", "") if parity else "",
    )
    if not match or int(match.group(1)) < 1 or int(match.group(2)) != 50:
        raise RuntimeError(
            f"Q8 first-token oracle failed: {parity[-1] if parity else None}"
        )
    oracle_matches = int(match.group(1))
    measurements = data.get("measurements") or {}
    count, prefill_ms, decode_ms = timing_rows(
        measurements.get("iterations") or [], MEASURE_ITERS, "measured"
    )
    _, cold_prefill_ms, cold_decode_ms = timing_rows(
        measurements.get("cold") or [], COLD_ITERS, "cold"
    )
    median_latency = statistics.median(prefill_ms)
    throughput = [count * 1000.0 / value for value in prefill_ms]
    return {
        "prefill_p50": percentile(throughput, 0.50),
        "prefill_p90": percentile(throughput, 0.90),
        "prefill_p99": percentile(throughput, 0.99),
        "latency_p50_ms": percentile(prefill_ms, 0.50),
        "latency_p90_ms": percentile(prefill_ms, 0.90),
        "latency_p99_ms": percentile(prefill_ms, 0.99),
        "latency_mad_ms": statistics.median(
            abs(value - median_latency) for value in prefill_ms
        ),
        "latency_max_ms": max(prefill_ms),
        "decode_p50": percentile([1000.0 / value for value in decode_ms], 0.50),
        "cold_prefill_ms": cold_prefill_ms[0],
        "cold_decode_ms": cold_decode_ms[0],
        "samples_ms": [round(value, 6) for value in prefill_ms],
        "oracle": f"first-token exact; {oracle_matches}/50 decode prefix",
        "oracle_matches": oracle_matches,
    }


try:
    for key in list(os.environ):
        if key.startswith("GLCUDA_"):
            os.environ.pop(key)

    target = ROOT / "target-wave102"
    build_env = {**cargo_env, "CARGO_TARGET_DIR": str(target)}
    build = run(
        [cargo, "build", "--release", "-p", "glbench", "--locked"],
        cwd=TREE,
        env=build_env,
        timeout=7200,
        check=False,
    )
    save("cargo-build-glbench.log", build)
    if build.returncode:
        raise RuntimeError("Wave 102 glbench release build failed")
    glbench = target / "release/glbench"

    model_path = fetch_pinned_model()
    model_meta = {
        "repo": HF_REPO,
        "revision": HF_REVISION,
        "filename": HF_FILENAME,
        "bytes": model_path.stat().st_size,
        "sha256": sha256_file(model_path),
    }
    (RESULTS / "model.json").write_text(json.dumps(model_meta, indent=2), encoding="utf-8")

    common_env = {
        **cargo_env,
        "CUDA_VISIBLE_DEVICES": "0",
        "CARGO_TARGET_DIR": str(target),
        "GLCUDA_FORCE_Q8": "1",
        "GLCUDA_GRID2D": "1",
        "GLCUDA_FUSE_Q8_GLUE": "1",
        "GLCUDA_NTILE128": "1",
        "GLCUDA_BSTAGE": "1",
        "GLCUDA_GEMM_N16": "1",
        "GLCUDA_ATTN_MMA4": "1",
        "GLCUDA_ATTN_MMA4_REGQ": "1",
        "GLCUDA_ATTN_MMA4_AV": "1",
    }
    arm_env = {
        "retained_n16": {},
        "candidate_n16_prefetch": {"GLCUDA_GEMM_N16_PREFETCH": "1"},
    }
    orders = [
        ["retained_n16", "candidate_n16_prefetch"]
        if repeat % 2 == 0
        else ["candidate_n16_prefetch", "retained_n16"]
        for repeat in range(PRODUCTION_REPEATS)
    ]

    def run_arm(arm, output, cold, warmup, iterations):
        env = {**common_env, **arm_env[arm]}
        command = [
            glbench,
            "run",
            "--engine", "glcuda",
            "--model", model_path,
            "--prompt", FIXED_PROMPT,
            "--tokens", "1",
            "--cold-iters", str(cold),
            "--warmup", str(warmup),
            "--iters", str(iterations),
            "--temperature", "0",
            "--seed", "42",
            "--kind", "prefill",
            "--verify-against", "glproc",
            "--out", output,
        ]
        return run(command, cwd=TREE, env=env, timeout=14400, check=False)

    dispatch = {}
    for arm in arm_env:
        process = run_arm(arm, RESULTS / f"stabilize-{arm}.json", 0, 0, 1)
        save(f"stabilize-{arm}.log", process)
        if process.returncode:
            raise RuntimeError(f"stabilization failed for {arm}")
        dispatch[arm] = check_dispatch(arm, process.stdout + "\n" + process.stderr)

    records = []
    for repeat, order in enumerate(orders):
        for position, arm in enumerate(order):
            output = RESULTS / f"glbench-r{repeat}-p{position}-{arm}.json"
            process = run_arm(arm, output, COLD_ITERS, WARMUP_ITERS, MEASURE_ITERS)
            save(f"glbench-r{repeat}-p{position}-{arm}.log", process)
            if process.returncode:
                raise RuntimeError(f"glbench failed: {arm} repeat {repeat}")
            check_dispatch(arm, process.stdout + "\n" + process.stderr)
            stats = session_stats(output)
            records.append({
                "repeat": repeat,
                "position": position,
                "arm": arm,
                **stats,
            })
            print(
                f"{arm:16s} r{repeat} p{position}: {stats['prefill_p50']:8.1f} tok/s | "
                f"P50/P90/P99 {stats['latency_p50_ms']:.3f}/"
                f"{stats['latency_p90_ms']:.3f}/{stats['latency_p99_ms']:.3f} ms",
                flush=True,
            )

    summary = {}
    for arm in arm_env:
        rows = [row for row in records if row["arm"] == arm]
        summary[arm] = {
            "prefill_p50_median": statistics.median(row["prefill_p50"] for row in rows),
            "prefill_p90_median": statistics.median(row["prefill_p90"] for row in rows),
            "prefill_p99_median": statistics.median(row["prefill_p99"] for row in rows),
            "latency_p50_median_ms": statistics.median(row["latency_p50_ms"] for row in rows),
            "latency_p90_median_ms": statistics.median(row["latency_p90_ms"] for row in rows),
            "latency_p99_median_ms": statistics.median(row["latency_p99_ms"] for row in rows),
            "latency_mad_median_ms": statistics.median(row["latency_mad_ms"] for row in rows),
            "latency_max_median_ms": statistics.median(row["latency_max_ms"] for row in rows),
            "decode_p50_median": statistics.median(row["decode_p50"] for row in rows),
            "sessions": len(rows),
            "positions": [row["position"] for row in rows],
        }

    paired = []
    for repeat in range(PRODUCTION_REPEATS):
        retained = next(
            row for row in records
            if row["repeat"] == repeat and row["arm"] == "retained_n16"
        )
        candidate = next(
            row for row in records
            if row["repeat"] == repeat and row["arm"] == "candidate_n16_prefetch"
        )
        paired.append({
            "repeat": repeat,
            "throughput_delta": candidate["prefill_p50"] / retained["prefill_p50"] - 1.0,
            "absolute_tps": candidate["prefill_p50"] - retained["prefill_p50"],
            "tail_max_delta": candidate["latency_max_ms"] / retained["latency_max_ms"] - 1.0,
            "decode_delta": candidate["decode_p50"] / retained["decode_p50"] - 1.0,
        })

    retained = summary["retained_n16"]
    candidate = summary["candidate_n16_prefetch"]
    median_ratio = candidate["prefill_p50_median"] / retained["prefill_p50_median"] - 1.0
    absolute_gain = candidate["prefill_p50_median"] - retained["prefill_p50_median"]
    tail_delta = candidate["latency_max_median_ms"] / retained["latency_max_median_ms"] - 1.0
    oracle_ok = all(row["oracle_matches"] >= 1 for row in records)
    decode_ok = all(row["decode_delta"] >= -0.05 for row in paired)
    tail_ok = tail_delta <= 0.05
    all_positive = all(row["throughput_delta"] > 0 for row in paired)
    retention_ok = median_ratio >= 0.01 and all_positive and decode_ok and tail_ok
    target_reached = candidate["prefill_p50_median"] >= 15_000.0
    if target_reached and retention_ok:
        decision = "GOAL_REACHED"
        verdict = "RETAIN - production gate passes and reaches 15,000 tok/s"
    elif retention_ok:
        decision = "RETAIN_BELOW_GOAL"
        verdict = "RETAIN - production gate passes, still below 15,000 tok/s"
    else:
        decision = "REJECT"
        verdict = "REJECT - candidate misses the production retention gate"

    result = {
        "wave": 80,
        "status": "production_measured",
        "gpu": gpu_line,
        "source_revision": SOURCE_REV,
        "patch_sha256": PATCH_SHA256,
        "model": model_meta,
        "direct_gate": direct_result,
        "method": {
            "repeats": PRODUCTION_REPEATS,
            "sessions": len(records),
            "cold_iters": COLD_ITERS,
            "warmup_iters": WARMUP_ITERS,
            "measure_iters": MEASURE_ITERS,
            "order": orders,
            "quant": "Q8_0",
            "prompt_tokens": 244,
            "seed": 42,
            "temperature": 0.0,
            "oracle": "glproc first-token exact; contiguous decode prefix recorded",
        },
        "summary": summary,
        "paired": paired,
        "comparison": {
            "ratio_of_session_p50_medians": median_ratio,
            "absolute_gain_tps": absolute_gain,
            "median_paired_delta": statistics.median(row["throughput_delta"] for row in paired),
            "worst_paired_delta": min(row["throughput_delta"] for row in paired),
            "all_positive": all_positive,
            "tail_session_max_delta": tail_delta,
            "oracle_ok": oracle_ok,
            "decode_ok": decode_ok,
            "tail_ok": tail_ok,
        },
        "dispatch": dispatch,
        "decision": decision,
        "verdict": verdict,
        "target_15000_tps_achieved": target_reached and retention_ok,
    }
    (RESULTS / "production-records.json").write_text(
        json.dumps(records, indent=2), encoding="utf-8"
    )
    (RESULTS / "wave102-production.json").write_text(
        json.dumps(result, indent=2), encoding="utf-8"
    )
    (RESULTS / "PRODUCTION_SUCCESS.json").write_text(
        json.dumps({"status": "valid", "decision": decision}, indent=2),
        encoding="utf-8",
    )
    archive()
    print(json.dumps(result, indent=2))
except Exception:
    (RESULTS / "PRODUCTION_FAILED.txt").write_text(
        traceback.format_exc(), encoding="utf-8"
    )
    archive()
    raise
